# ?lections au Conseil national ? exploration

Lire les tables cr??es par `python -m scripts.run_pipeline --all`. Les codes sont harmonis?s au 01.01.2026 ; les publications historiques restent non v?rifi?es.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Fonctionne depuis la racine ou le dossier du notebook.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").is_file())

source = pd.read_parquet(ROOT / "data/interim/elections/elections_communes_2023_geography.parquet")
elections = pd.read_parquet(ROOT / "data/processed/elections/elections_communes.parquet")
display(elections.head())

## Couverture et pr?sence des partis

In [ ]:
ordinary = elections.loc[~elections.is_special_unit]
display(ordinary.groupby("election_year").municipality_id.nunique().rename("communes"))
display(ordinary.pivot_table(index="party_name", columns="election_year", values="party_share", aggfunc="count"))

## Donn?es manquantes et unicit?

Les absences ne sont pas remplac?es par z?ro. PDC, PBD et Centre restent distincts.

In [ ]:
display(ordinary.isna().mean().sort_values(ascending=False))
print("Duplicate keys:", elections.duplicated(["municipality_id", "election_year", "party_code"]).sum())

## Contr?le des pourcentages dans la g?ographie source

Les communes fusionn?es peuvent conserver des valeurs manquantes partielles : v?rifier les sommes avant harmonisation et les compteurs de valeurs manquantes apr?s.

In [ ]:
totals = source.groupby(["municipality_id", "election_year"]).party_strength_pct.sum(min_count=1)
display(totals.describe())
print("Source totals outside rounding tolerance:", ((totals - 100).abs() > .2).sum())
display(ordinary.source_values_missing.value_counts().sort_index())